In [1]:
from __future__ import division
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
#from keras.applications.vgg16 import VGG16
#from keras.applications.vgg19 import VGG19
#from keras.applications.resnet import ResNet50
#from keras.applications.resnet import ResNet101
from keras.applications.resnet import ResNet152
#from keras.applications.resnet_v2 import ResNet50V2
#from keras.applications.resnet_v2 import ResNet101V2
#from keras.applications.resnet_v2 import ResNet152V2
from keras.applications.inception_resnet_v2 import InceptionResNetV2
#from keras.applications.inception_v3 import InceptionV3
#from keras.applications.densenet import DenseNet121
#from keras.applications.densenet import DenseNet169
#from keras.applications.densenet import DenseNet201
from keras.activations import softmax, relu, sigmoid
from keras.optimizers import SGD
from keras.utils.np_utils import to_categorical
from keras.callbacks import ModelCheckpoint,TensorBoard,ReduceLROnPlateau,CSVLogger, EarlyStopping
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import cv2
from random import shuffle
import random
import glob
from skimage.transform import resize
#from PIL import Image
import os
#from model import *

from collections import defaultdict
import itertools
from tqdm import tqdm
import time
import shutil

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


2024-08-01 14:11:25.550020: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2, in other operations, rebuild TensorFlow with the appropriate compiler flags.


ModuleNotFoundError: No module named 'skimage'

# Evaluating Image Classifiers

tentative plan:
- discuss simple evaluation metrics for classification; simple in image recognition context
- possibly discuss for image segmentation or object recognition?

In [ ]:
config = tf.ConfigProto()
config.gpu_options.allow_growth = True
config.log_device_placement = True
sess = tf.Session(config=config)
set_session(sess)

In [ ]:
model_name = 'Inception-ResNet-v2_Jan2020'
num_classes = 14

In [ ]:
input_shape=(256,256,3)
#model= VGG16(include_top=True, weights=None, input_tensor=None, input_shape=input_shape, pooling=None, classes=num_classes)
#model= VGG19(include_top=True, weights=None, input_tensor=None, input_shape=input_shape, pooling=None, classes=num_classes)
#model= ResNet50(include_top=True, weights=None, input_tensor=None, input_shape=input_shape, pooling=None, classes=num_classes)
#= ResNet101(include_top=True, weights=None, input_tensor=None, input_shape=input_shape, pooling=None, classes=num_classes)
#model= ResNet152(include_top=True, weights=None, input_tensor=None, input_shape=input_shape, pooling=None, classes=num_classes)
#model= ResNet50V2(include_top=True, weights=None, input_tensor=None, input_shape=input_shape, pooling=None, classes=num_classes)
#model= ResNet101V2(include_top=True, weights=None, input_tensor=None, input_shape=input_shape, pooling=None, classes=num_classes)
#model= ResNet152V2(include_top=True, weights=None, input_tensor=None, input_shape=input_shape, pooling=None, classes=num_classes)
model= InceptionResNetV2(include_top=True, weights=None, input_tensor=None, input_shape=input_shape, pooling=None, classes=num_classes)
#model= InceptionV3(include_top=True, weights=None, input_tensor=None, input_shape=input_shape, pooling=None, classes=num_classes)
#model= DenseNet121(include_top=True, weights=None, input_tensor=None, input_shape=input_shape, pooling=None, classes=num_classes)
#model= DenseNet169(include_top=True, weights=None, input_tensor=None, input_shape=input_shape, pooling=None, classes=num_classes)
#model= DenseNet201(include_top=True, weights=None, input_tensor=None, input_shape=input_shape, pooling=None, classes=num_classes)
#model =InceptionResNetV2(include_top=True, weights=None, input_tensor=None, input_shape=input_shape, pooling=None, classes=num_classes)
#model = resNet(input_shape, num_classes, model_type='resnet_152')
model.load_weights('WEIGHTS/Trained_Models/'+model_name+'.h5')

### model_vgg16.layers[-1].activation = relu

### idx_of_layer_to_change = -1
#model.layers[idx_of_layer_to_change].activation = activations.softmax
#model = utils.apply_modifications(model)

#model_vgg16.layers[-1].activation = relu
# # FINE TUNING HERE
# top_model = Sequential()
# top_model.add(Dense(input_shape=model.layers[-2].output_shape, units=num_classes, rnel
# kernel_initializer="he_normal", activation="softmax"))

# model.layers.pop()
# model.outputs = [model.layers[-1].output]
# model.layers[-1].outbound_nodes = []

# model = Model(inputs=model.inputs, outputs=top_model(model.outputs[0]))
# # for layer in model.layers[:-1]:
# #     layer.trainable = False
#model.load_weights('WEIGHTS/'+model_name+'.h5')

model.summary()
sgd = SGD(lr=0.01, decay=1e-6, momentum=0.9, nesterov=True)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])
# plot_model(model, to_file='model.png')

In [ ]:
test_images_dir = 'kotRadha/'
#test_images_dir = 'TestingDatasets/LahoreZ20LatLon/370320/'
#test_images_dir = 'TestingDatasets/kotRadhaTrain/'
#test_images_dir = 'TestingDatasets//Lahorez17toz20/370320/'
results_dir = 'Results/Pakistan/kotradha_ir/'
#results_dir = 'Results/Lahore15-10-19/'
#results_dir = 'Results/Lahorez20to20/370320/'
if not os.path.exists(results_dir):
    print('Creating new results directory "{}"'.format(results_dir))
    os.mkdir(results_dir)
im_dir = os.path.join(results_dir, 'images')
if not os.path.exists(im_dir):
    os.mkdir(im_dir)

In [ ]:
## print('Waiting'len(images)len(images)len(images), end='', flush=True)len(images)
# while True:
    
 #   if len(os.listdir(test_images_dir)) != 544244:
 #      time.sleep(10*60)
 #       print('.', end='', flush=True)
 #       continue
start_time = time.time()

# if os.path.exists(os.path.join(results_dir, 'houses.txt')) or os.path.exists(os.path.join(results_dir, 'test.csv')):
#     raise OSError('Result files already present, this script will append to the existing data.')
    
# label_names = {0: 'parking', 1: 'parks', 2: 'ground', 3: 'houses', 4: 'roads', 5: 'mosque', 6: 'densetrees', 7: 'kiln', 8: 'oiltanks', 9: 'tennis', 10: 'ponds', 11: 'grass', 12: 'blackfarms', 13: 'farms', 14: 'orchard'}
# label_names = {0: 'parking', 1: 'parks', 2: 'ground', 3: 'houses', 4: 'roads', 5: 'mosque', 6: 'densetrees', 7: 'kiln', 8: 'oiltanks', 9: 'tennis', 10: 'ponds', 11: 'grass', 12: 'blackfarms', 13: 'farms'}
# label_names = {0: 'blackfarms', 1: 'densetrees', 2: 'farms', 3: 'grass', 4: 'ground', 5: 'houses', 6: 'kiln', 7: 'mosque', 8: 'oiltanks', 9: 'orchard', 10: 'parking', 11: 'parks', 12: 'ponds', 13: 'roads', 14: 'tennis'}
label_names = {0: 'blackfarms', 
               1: 'densetrees', 
               2: 'farms', 
               3: 'grass', 
               4: 'ground', 
               5: 'houses', 
               6: 'kiln', 
               7: 'mosque', 
               8: 'oiltanks', 
               9: 'parking',
               10:'parks',
               11:'ponds',
               12:'roads',
               13:'tennis'}
y_tiles, x_tiles = [], []
label_probs = defaultdict(list)
top_label_probs = defaultdict(list)

# # Resuming check
# try:
#     with open(os.path.join(results_dir, 'last_file_done.txt'), 'r') as f:
#         loop_index = int(f.read())
#     print('Resuming from file index "{}"'.format(loop_index))
# except FileNotFoundError:
#     loop_index = 0
#     print('Starting fresh from file index "0"')

filenames = os.listdir(test_images_dir)

# # Missing images check
# try:
#     with open(os.path.join(results_dir, 'missing_images.txt'), 'r') as f:
#         missing_images = [name.strip() for name in f.readlines()]
#     print('Continuing with {} missing images.'.format(len(missing_images)))
# except FileNotFoundError:
#     missing_images = []
#     print('Starting fresh with empty missing images list')
for img_name in tqdm(filenames):


#     extt=img_name.split('.')[1]
    orig_name, extt = os.path.splitext(img_name)
    if(extt == '.jpg'):
        img_orig = cv2.imread(os.path.join(test_images_dir, img_name), cv2.IMREAD_COLOR)
        # To catch corrupt images
        if type(img_orig) == type(None):
            print('Skipping an image "{}"'.format(img_name))
        #    missing_images.append(img_name)
            continue

        img = np.expand_dims(img_orig, axis=0)
        class_name = img_name.split('.')[0]

        pred = model.predict(img, verbose=0)
    #    preds.append((pred.argmax(), pred.max(), class_name))

#         src = os.path.join(test_images_dir, img_name)
#         dst = os.path.join(results_dir, 'images', orig_name+'_'+label_names[pred.argmax()]+extt)
#         shutil.copyfile(src, dst)
#         print(img_name, dst)

#         break
        x_tile, y_tile = os.path.splitext(img_name)[0].split('_')
        y_tiles.append(y_tile)
        x_tiles.append(x_tile)

        # Append to individual results files on each itteration
        for i, label in label_names.items():
            file_path = os.path.join(results_dir, label+'.txt')
            label_probs[label].append(pred[0, i])
            with open(file_path, 'a') as f:
                f.write('{} {} {}\n'.format(y_tile, x_tile, pred[0, i]))
#         # Append to full result file on every 1000 itterations
#         #if loop_index % 1000 == 0:
#         d = {'y_tile': y_tiles, 'x_tile': x_tiles}
#         d.update(label_probs)
#         final_df = pd.DataFrame(data=d)
#         final_df.to_csv(os.path.join(results_dir, 'test.csv'), index=False, header=True)
        # Resuming 
        #with open('last_file_done.txt', 'w') as f:
         #   f.write(str(loop_index))
        # Write names of corrupted files for later use
        #with open(os.path.join(results_dir, 'missing.txt'), 'w') as f:
           # for i in missing_images:
           #     f.write('{}\n'.format(i))
        #loop_index += 1

        top3_inds = np.argsort(pred)
        top3_inds = np.flip(top3_inds, axis=1)
        top3_labels = [ label_names[i] for ia in top3_inds for i in ia ]
        top3_probs = pred[0, top3_inds]
#     print(top3_labels)

        for n in range(num_classes):
            top_label_probs['label'+str(n+1)].append(top3_labels[n])
            top_label_probs['prob'+str(n+1)].append(top3_probs[0, n])

# Append to full result file last time
d = {'y_tile': y_tiles, 'x_tile': x_tiles}
d.update(label_probs)
final_df = pd.DataFrame(data=d)
final_df.to_csv(os.path.join(results_dir, 'test.csv'), index=False, header=True)
# final_df

# Write names of corrupted files for later use
#with open(os.path.join(results_dir, 'missing.txt'), 'w') as f:
#    for i in missing_images:
#        f.write('{}\n'.format(i))

d = {'y_tile': y_tiles, 'x_tile': x_tiles}
d.update(top_label_probs)
sorted_df = pd.DataFrame(data=d)
sorted_df = sorted_df[['y_tile', 'x_tile', 'label1', 'prob1', 'label2', 'prob2', 'label3', 'prob3', 'label4', 'prob4', 'label5', 'prob5', 'label6', 'prob6', 'label7', 'prob7', 'label8', 'prob8', 'label9', 'prob9', 'label10', 'prob10','label11', 'prob11','label12', 'prob12','label13', 'prob13','label14', 'prob14']]
sorted_df.to_csv(os.path.join(results_dir, 'top_predictions.csv'), index=False, header=True)
print("Seconds: ", time.time() - start_time)
# sorted_df.head()

# preds = sorted(preds, key=lambda x: x[0])
# for p_class, p_prob, truth in preds:
#     got_it = label_names[p_class] == truth
#     print('{}\tPrediction: {}\tTruth: {}\tProb: {}'.format(got_it, label_names[p_class], truth, p_prob))
#     break